# Real vLLM inference with TRITON_ATTN

**EXPERIMENTAL — KAGGLE DUAL-T4 VALIDATION PENDING.** Not executed by the author; outputs are produced only when *you* run it.

Accelerator: **GPU T4 x2**, Internet **ON** (notebooks 02–06 download the 8 pinned helper wheels, ~57 MB, and 05/06 download the model). Run notebooks in order 01 → 06 (each Kaggle notebook is a fresh session, so notebooks 03–06 re-run the idempotent installer first); each stage writes JSON to `/kaggle/working/stokastisk-acceptance/`.

Loads `Qwen/Qwen2.5-0.5B-Instruct` (public, ~1 GB fp16) from Hugging Face, runs short-decode and ~1.2k-token prefill generations at TP=1 and TP=2, checks determinism, and **greps vLLM's own log** for the selected backend. Set `HF_TOKEN` as a Kaggle secret only for gated models.

## 1. Setup — locate the release bundle

In [ ]:
import glob, os, pathlib
# Locate the release bundle (the Kaggle dataset holding manifests/SHA256SUMS). Override with STOKASTISK_BUNDLE.
hits = [pathlib.Path(p).parent.parent for p in glob.glob("/kaggle/input/**/manifests/SHA256SUMS", recursive=True)]
BUNDLE = pathlib.Path(os.environ.get("STOKASTISK_BUNDLE") or (hits[0] if hits else "")) 
assert (BUNDLE / "manifests" / "SHA256SUMS").is_file(), "attach the stokastisk-0.3.0 release dataset to this notebook"
SCRIPTS = BUNDLE / "scripts" / "kaggle"
os.environ["STOKASTISK_OUT"] = "/kaggle/working/stokastisk-acceptance"
print("bundle:", BUNDLE)

## 2. Install into this fresh Kaggle session (idempotent, `--no-deps`, SHA256-verified, protected packages untouched)

Kaggle gives every notebook a new session: what notebook 02 installed does not exist here. Internet must be ON.

In [ ]:
import subprocess, sys, os
r = subprocess.run([sys.executable, str(SCRIPTS / "02_install.py"), str(BUNDLE)],
                   env=dict(os.environ, STOKASTISK_OUT="/kaggle/working/stokastisk-acceptance/prereq-install"))
assert r.returncode == 0, "prerequisite install failed (see output above); do not continue"

## 3. Provenance — every executed script and wheel must match `manifests/SHA256SUMS`

In [ ]:
import hashlib, glob
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
print("bundle root     :", BUNDLE)
print("scripts/kaggle  :", SCRIPTS)
for s in sorted(SCRIPTS.glob("*.py")): print(f"{sha(s)[:16]}  {s.name}")
for w in sorted((BUNDLE / "wheels").glob("*.whl")): print(f"{sha(w)}  {w.name}")
expected = {l.split(None, 1)[1].strip(): l.split()[0] for l in (BUNDLE / "manifests" / "SHA256SUMS").read_text().splitlines()}
for rel in [f"scripts/kaggle/{s.name}" for s in SCRIPTS.glob("*.py")] + [f"wheels/{w.name}" for w in (BUNDLE / "wheels").glob("*.whl")]:
    assert expected.get(rel) == sha(BUNDLE / rel), f"{rel} does not match manifests/SHA256SUMS (stale or modified file)"
print("all executed scripts and wheels match manifests/SHA256SUMS")

## 4. The installed packages must be byte-identical to the bundle's wheels

In [ ]:
import importlib, importlib.metadata as md, zipfile, hashlib
for w in sorted((BUNDLE / "wheels").glob("*.whl")):
    pkg = w.name.split("-")[0]
    mod = importlib.import_module(pkg)
    root = pathlib.Path(mod.__file__).resolve().parent
    z = zipfile.ZipFile(w); bad = []
    for n in z.namelist():
        if n.startswith(pkg + "/") and n.endswith((".py", ".so")):
            f = root.parent / n
            if not f.is_file() or hashlib.sha256(f.read_bytes()).hexdigest() != hashlib.sha256(z.read(n)).hexdigest(): bad.append(n)
    print(f"{pkg} {md.version(pkg)} imported from {root}: {'identical to ' + w.name if not bad else 'MISMATCH ' + str(bad)}")
    assert not bad, f"installed {pkg} differs from the bundle wheel {w.name}; uninstall it and re-run the install cell"

## 5. Run Qwen2.5-0.5B-Instruct at TP=1 and TP=2

In [ ]:
import subprocess, sys, os
env = dict(os.environ, PYTHONPATH=str(SCRIPTS) + ":" + os.environ.get("PYTHONPATH", ""), VLLM_LOGGING_LEVEL="INFO")
for tp in (1, 2):
    with open(f"/kaggle/working/stokastisk-acceptance/vllm_tp{tp}.log", "w") as f:
        rc = subprocess.run([sys.executable, str(SCRIPTS / "05_inference.py"), str(tp), "Qwen/Qwen2.5-0.5B-Instruct"],
                            env=env, stdout=f, stderr=subprocess.STDOUT).returncode
    print("TP", tp, "rc", rc)

## 6. Prove from vLLM's own logs that `TRITON_ATTN` was selected

In [ ]:
import re, json
OUT = "/kaggle/working/stokastisk-acceptance"
ev = {}
for tp in (1, 2):
    log = open(f"{OUT}/vllm_tp{tp}.log", errors="replace").read()
    chosen = re.findall(r"Using (?:AttentionBackendEnum\.)?TRITON_ATTN backend", log)
    other = re.findall(r"(?i)using[^\n]*(?:FLASH_?ATTN|FLASHINFER|FLASHMLA|FLEX_ATTENTION|XFORMERS)[^\n]*backend", log)
    ev[tp] = {"triton_lines": len(chosen), "other_backend_lines": other[:3], "log_bytes": len(log)}
    print(f"TP{tp}: {ev[tp]}")
    assert len(chosen) >= tp, f"expected >= {tp} 'Using ...TRITON_ATTN backend' log line(s) (one per worker); inspect {OUT}/vllm_tp{tp}.log"
    assert not other, "a non-Triton attention backend was reported"
    assert "Selected backend" not in log or "is not valid" not in log, "vLLM rejected the selected backend"
json.dump(ev, open(f"{OUT}/05_backend_log_evidence.json", "w"), indent=1)

In [ ]:
print(open("/kaggle/working/stokastisk-acceptance/05_inference_tp1.json").read()[:1500])

## Package the evidence

Creates `/kaggle/working/05-stokastisk-acceptance.zip` (download it from the Output tab).

In [ ]:
import shutil
print(shutil.make_archive("/kaggle/working/05-stokastisk-acceptance", "zip", "/kaggle/working", "stokastisk-acceptance"))